# Machine Learning Assignment - 2
**BITS Pilani WILP | M.Tech (AIML)**

Dataset: UCI Dry Bean (multi-class classification)  
Models: Logistic Regression, Decision Tree, KNN, Naive Bayes, SVM  
Run this notebook on **BITS Virtual Lab** and capture **one screenshot** for the submission PDF.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

print('Libraries imported successfully')
print('Working directory:', Path.cwd())

## 1. Load / download dataset

In [ ]:
data_path = Path('data/dry_bean_full.csv')
data_path.parent.mkdir(parents=True, exist_ok=True)

if data_path.exists():
    df = pd.read_csv(data_path)
    print('Loaded local CSV:', data_path)
else:
    try:
        from ucimlrepo import fetch_ucirepo
        dry_bean = fetch_ucirepo(id=602)
        df = pd.concat([dry_bean.data.features, dry_bean.data.targets], axis=1)
        if 'Class' not in df.columns:
            df = df.rename(columns={df.columns[-1]: 'Class'})
        print('Downloaded via ucimlrepo')
    except Exception as err:
        print('ucimlrepo failed:', err)
        url = 'https://www.stat.rice.edu/~scottdw/stat541/PROJECTS/uci-data/DRY-BEAN/Dry_Bean_Dataset.csv'
        df = pd.read_csv(url)
        df.columns = [c.strip() for c in df.columns]
        if 'Class' not in df.columns:
            df = df.rename(columns={df.columns[-1]: 'Class'})
        # Harmonize older mirror column names if present
        rename_map = {'AspectRation': 'AspectRatio', 'roundness': 'Roundness'}
        df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
        print('Downloaded via CSV mirror')
    df.to_csv(data_path, index=False)

df['Class'] = df['Class'].astype(str).str.strip()
print('Shape:', df.shape)
print('Features:', [c for c in df.columns if c != 'Class'])
display(df.head())
print('\nClass distribution:')
print(df['Class'].value_counts())

## 2. Train / test split

In [ ]:
feature_cols = [c for c in df.columns if c != 'Class']
X = df[feature_cols]
y_raw = df['Class']

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Train size:', X_train.shape[0], '| Test size:', X_test.shape[0])
print('Classes:', list(label_encoder.classes_))

## 3. Define five classifiers

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42)),
    ]),
    'Decision Tree': DecisionTreeClassifier(
        criterion='gini', max_depth=18, min_samples_leaf=4, random_state=42
    ),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=7, weights='distance')),
    ]),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=2.0, gamma='scale', probability=True, random_state=42)),
    ]),
}

list(models.keys())

## 4. Train, evaluate metrics, and compare

In [ ]:
def evaluate(model, X_te, y_te, classes):
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)
    return {
        'Accuracy': accuracy_score(y_te, pred),
        'AUC': roc_auc_score(y_te, proba, multi_class='ovr', average='weighted',
                             labels=list(range(len(classes)))),
        'Precision': precision_score(y_te, pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_te, pred, average='weighted', zero_division=0),
        'F1': f1_score(y_te, pred, average='weighted', zero_division=0),
        'MCC': matthews_corrcoef(y_te, pred),
    }, pred

Path('model').mkdir(exist_ok=True)
Path('artifacts').mkdir(exist_ok=True)

all_metrics = {}
for name, model in models.items():
    print('\n===', name, '===')
    model.fit(X_train, y_train)
    metrics, y_pred = evaluate(model, X_test, y_test, label_encoder.classes_)
    all_metrics[name] = metrics
    joblib.dump(model, Path('model') / f"{name.lower().replace(' ', '_')}.joblib")
    for k, v in metrics.items():
        print(f'{k:10s}: {v:.4f}')
    print('\nClassification report:')
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_, ax=ax)
    ax.set_title(f'Confusion Matrix - {name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.show()

joblib.dump(label_encoder, 'model/label_encoder.joblib')
joblib.dump(feature_cols, 'model/feature_columns.joblib')

test_export = X_test.copy()
test_export['Class'] = label_encoder.inverse_transform(y_test)
test_export.to_csv('test_data.csv', index=False)

comparison = pd.DataFrame(all_metrics).T[['Accuracy', 'AUC', 'Precision', 'Recall', 'F1', 'MCC']]
print('\n===== COMPARISON TABLE =====')
display(comparison.round(4))
print('\nWinner:', comparison['Accuracy'].idxmax())

with open('artifacts/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=2)

print('\nSaved models to model/, test_data.csv, and artifacts/metrics.json')
print('Take your BITS Virtual Lab screenshot of this comparison table cell output.')

## 5. Quick check — load a saved model (no retraining)

In [ ]:
loaded = joblib.load('model/support_vector_machine.joblib')
sample_pred = loaded.predict(X_test.iloc[:5])
print('Sample predictions:', label_encoder.inverse_transform(sample_pred))
print('Actual labels     :', label_encoder.inverse_transform(y_test[:5]))